# 🧠 Behavior Classifier Training (Optional)

This trains a tiny sklearn classifier on top of the rule-based system.
Use GPU runtime for DeepFace speed-up during data collection.

**Runtime → Change runtime type → T4 GPU**

This is OPTIONAL — the rule-based `app.py` works without this.

In [ ]:
# Install dependencies
!pip install deepface mediapipe tf-keras scikit-learn joblib opencv-python-headless -q

In [ ]:
import numpy as np
import math
import cv2
import mediapipe as mp
from deepface import DeepFace
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
import json

In [ ]:
# ── Feature extraction (same as app.py) ──────────────────────────

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

EMOTIONS = ['happy', 'sad', 'angry', 'fear', 'surprise', 'disgust', 'neutral']

def emotion_to_vec(scores: dict) -> list:
    return [scores.get(e, 0.0) for e in EMOTIONS]

def extract_posture_features(landmarks):
    lm = landmarks.landmark
    def pt(idx): return np.array([lm[idx].x, lm[idx].y, lm[idx].z])
    def angle(a, b, c):
        ba, bc = a-b, c-b
        cos_a = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
        return math.degrees(math.acos(np.clip(cos_a,-1,1)))

    nose=pt(0); l_sh=pt(11); r_sh=pt(12)
    l_hip=pt(23); r_hip=pt(24)
    l_ear=pt(7); r_ear=pt(8)
    l_el=pt(13); r_el=pt(14)
    l_wr=pt(15); r_wr=pt(16)

    mid_sh = (l_sh+r_sh)/2
    mid_hip = (l_hip+r_hip)/2
    sh_tilt = abs(l_sh[1]-r_sh[1])
    head_fwd = nose[0]-mid_sh[0]
    spine_vec = mid_sh - mid_hip
    spine_ang = math.degrees(math.atan2(abs(spine_vec[0]), abs(spine_vec[1])+1e-6))
    sh_width = abs(l_sh[0]-r_sh[0])
    avg_wr_y = (l_wr[1]+r_wr[1])/2
    arm_raise = mid_sh[1]-avg_wr_y
    head_tilt = abs(l_ear[1]-r_ear[1])
    elbow_spread = abs(l_el[0]-r_el[0])
    openness = elbow_spread/(sh_width+1e-6)
    slouch = nose[1]-mid_sh[1]

    return [sh_tilt, head_fwd, spine_ang, sh_width, arm_raise, head_tilt, openness, slouch]

def process_image(img_bgr):
    """Returns (feature_vector, label_hint) or None."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Emotion
    try:
        r = DeepFace.analyze(rgb, actions=['emotion'], enforce_detection=False, silent=True)
        if isinstance(r, list): r = r[0]
        emo_vec = emotion_to_vec(r['emotion'])
        dom_emo = r['dominant_emotion']
    except:
        emo_vec = [0.0]*7
        dom_emo = 'neutral'

    # Pose
    res = pose.process(rgb)
    if not res.pose_landmarks:
        return None

    pose_vec = extract_posture_features(res.pose_landmarks)
    feature = emo_vec + pose_vec  # 15-dim vector
    return feature, dom_emo

print('Feature extractor ready ✅')

In [ ]:
# ── Generate synthetic training data (no camera needed) ──────────
# We generate feature vectors matching known behavior rules.
# Replace with real captured data for better accuracy.

np.random.seed(42)

BEHAVIORS = [
    'Engaged & Attentive', 'Stressed / Anxious', 'Disengaged / Bored',
    'Confident & Expressive', 'Defensive', 'Excited / Animated',
    'Sad / Withdrawn', 'Thinking / Reflective', 'Frustrated', 'Calm & Relaxed'
]

EMO_IDX = {e: i for i, e in enumerate(EMOTIONS)}

def make_sample(dom_emo, sh_tilt, head_fwd, spine_ang, sh_width,
                arm_raise, head_tilt, openness, slouch, noise=0.05):
    emo_vec = [0.0]*7
    emo_vec[EMO_IDX[dom_emo]] = 0.7 + np.random.rand()*0.3
    for i in range(7):
        if i != EMO_IDX[dom_emo]:
            emo_vec[i] = np.random.rand()*0.1
    pose_vec = [sh_tilt, head_fwd, spine_ang, sh_width,
                arm_raise, head_tilt, openness, slouch]
    pose_vec = [v + np.random.randn()*noise for v in pose_vec]
    return emo_vec + pose_vec

N = 300  # samples per class
X, y = [], []

templates = [
    ('Engaged & Attentive',    'happy',    0.02, 0.0, 5.0,  0.3, 0.02, 0.02, 0.9,  0.05),
    ('Stressed / Anxious',     'fear',     0.04, 0.0, 8.0,  0.25,0.01, 0.02, 0.5,  0.1),
    ('Disengaged / Bored',     'neutral',  0.03, 0.0, 5.0,  0.28,0.01, 0.02, 0.8,  0.2),
    ('Confident & Expressive', 'happy',    0.02, 0.0, 4.0,  0.35,0.03, 0.03, 1.4,  0.04),
    ('Defensive',              'angry',    0.03, 0.0, 6.0,  0.22,0.01, 0.02, 0.45, 0.08),
    ('Excited / Animated',     'surprise', 0.04, 0.0, 5.0,  0.32,0.15, 0.03, 1.1,  0.05),
    ('Sad / Withdrawn',        'sad',      0.03, 0.0, 5.0,  0.26,0.01, 0.02, 0.55, 0.18),
    ('Thinking / Reflective',  'neutral',  0.02, 0.0, 5.0,  0.29,0.01, 0.07, 0.85, 0.06),
    ('Frustrated',             'angry',    0.04, 0.0,18.0,  0.3, 0.02, 0.03, 1.1,  0.08),
    ('Calm & Relaxed',         'neutral',  0.02, 0.0, 4.0,  0.3, 0.02, 0.02, 0.9,  0.06),
]

for behavior, *params in templates:
    for _ in range(N):
        X.append(make_sample(*params))
        y.append(behavior)

X = np.array(X)
y = np.array(y)
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')

In [ ]:
# ── Train classifier ─────────────────────────────────────────────
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# ── Save model ───────────────────────────────────────────────────
joblib.dump(clf, 'behavior_clf.joblib')
joblib.dump(le, 'label_encoder.joblib')

# Save feature names for reference
feature_names = EMOTIONS + ['shoulder_tilt','head_forward','spine_angle',
                             'shoulder_width','arm_raise','head_tilt','openness','slouch_score']
with open('feature_names.json','w') as f:
    json.dump(feature_names, f)

print('✅ Saved: behavior_clf.joblib, label_encoder.joblib')
print('Download these and place them in your project folder.')

In [ ]:
# ── Download files from Colab ────────────────────────────────────
from google.colab import files
files.download('behavior_clf.joblib')
files.download('label_encoder.joblib')

## Using the trained model in app.py

After downloading, place both `.joblib` files in your project folder and add this to `app.py`:

```python
import joblib
clf = joblib.load('behavior_clf.joblib')
le  = joblib.load('label_encoder.joblib')

def infer_behavior_ml(emotion_scores, posture_features):
    EMOTIONS = ['happy','sad','angry','fear','surprise','disgust','neutral']
    emo_vec = [emotion_scores.get(e, 0.0) for e in EMOTIONS]
    pose_vec = list(posture_features.values())
    feat = np.array([emo_vec + pose_vec])
    pred = clf.predict(feat)[0]
    proba = clf.predict_proba(feat)[0].max()
    return le.inverse_transform([pred])[0], float(proba), ''
```

Then swap `infer_behavior(...)` calls with `infer_behavior_ml(...)`.